In [1]:
!pip freeze | grep scikit-learn

scikit-learn @ file:///tmp/build/80754af9/scikit-learn_1642617106979/work
scikit-learn-intelex==2021.20220215.212715


In [2]:
!python -V

Python 3.9.12


In [3]:
import pickle
import pandas as pd

**Load Model**

In [4]:
def load_model(filepath):
    with open(filepath, 'rb') as f_in:
        dv, model = pickle.load(f_in)
    return dv, model

**Read Data**

In [5]:
def read_data(filename):
    categorical = ['PULocationID', 'DOLocationID']
    
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [ ]:
year = 2023
month = 3

df = read_data(f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet')

**Making Predictions**

In [8]:
categorical = ['PULocationID', 'DOLocationID']
dicts = df[categorical].to_dict(orient='records')

dv, model = load_model('model.bin')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

# Checking the standard deviation of predictions
print(f'Standard deviation of predictions: {y_pred.std():.2f}')

/home/codespace/anaconda3/lib/python3.9/site-packages/sklearn/base.py:329: UserWarning: Trying to unpickle estimator DictVectorizer from version 1.5.0 when using version 1.0.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/modules/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/codespace/anaconda3/lib/python3.9/site-packages/sklearn/base.py:329: UserWarning: Trying to unpickle estimator LinearRegression from version 1.5.0 when using version 1.0.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/modules/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Standard deviation of predictions: 6.25


**Saving the output**

In [9]:
df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

# Save the ride_id and predictions to a parquet file
df_result = pd.DataFrame(
    {'ride_id': df['ride_id'], 'prediction': y_pred}
)

output_file = f'yellow_tripdata_{year:04d}-{month:02d}_predictions.parquet'
df_result.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)

print(f'Results saved to {output_file}')

Results saved to yellow_tripdata_2023-03_predictions.parquet
